# Outlier testing against the existing model

This notebook compares each verified charge against the legacy sentencing model and highlights charges where the gap is large enough to merit manual review.

In [ ]:
from __future__ import annotations

import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from dotenv import load_dotenv
from tqdm.auto import tqdm

repo_root = Path.cwd().resolve()
if not (repo_root / 'featureExtraction').exists():
    repo_root = repo_root.parent

for env_path in (repo_root / 'featureExtraction' / '.env', repo_root / 'featureVerification' / '.env.local', repo_root / '.env'):
    if env_path.exists():
        load_dotenv(env_path)

from evaluate_verified_sentences import (
    ASSISTANCE_FACTOR_TO_INPUT,
    build_model_input,
    get_collection,
    get_total_months,
)
PREDICT_ENDPOINT = 'https://ai.hklii.hk/dt-predictor-backend/api/predict'


def predict_online(model_input: list[float]) -> dict[str, Any]:
    """Call the hosted legacy model endpoint and return the prediction dict.

    The endpoint accepts the 17-element model input vector as a JSON-array
    query parameter and returns the same structure as DkPredictor.explain()
    (starting_point, mitigating/aggravating factors, sentence_after_trial,
    final_sentence). The response body is double-encoded: the outer JSON has
    a success flag and a data field whose value is a JSON string, so data
    is parsed a second time.
    """
    response = requests.get(
        PREDICT_ENDPOINT,
        params={'predict': json.dumps(model_input)},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    if not payload.get('success'):
        raise RuntimeError(f'Predict endpoint returned success=false: {payload}')
    return json.loads(payload['data'])


verified_collection, _ = get_collection()
query = {'is_verified': True}
projection = {'filename': 1, 'judgement.neutral_citation': 1, 'exclude': 1, 'remarks': 1, 'trials': 1}
docs = list(verified_collection.find(query, projection))

OUTLIER_THRESHOLD_MONTHS = 6

# Ordered names for the 17-element legacy model input vector.
# Mirrors DkPredictor.explain() in legacy_model.py so each row exposes
# every value actually fed into the legacy model.
MODEL_INPUT_COLUMNS = [
    'self_consume',
    'assist_authorities',
    'other_mitigating',
    'refugee',
    'bail',
    'persistent',
    'international',
    'cocaine_amount',
    'heroin_amount',
    'meth_amount',
    'ketamine_amount',
    'nimetazepam_amount',
    'ecstasy_amount',
    'cannabisresin_amount',
    'herbalcannabis_amount',
    'plea_early',
    'plea_late',
]

def format_drugs(trial: dict[str, Any]) -> str:
    parts = []
    for drug in trial.get('drugs') or []:
        drug_type = drug.get('drug_type')
        quantity = drug.get('quantity')
        if drug_type:
            parts.append(f"{drug_type}:{quantity}")
    return '; '.join(parts)

def get_assistance_flags(trial: dict[str, Any]) -> dict[str, bool]:
    """Return a boolean for each assistance-to-authorities factor type.

    The legacy model collapses these into a single ordinal value
    (assist_authorities = max severity). Exposing each type lets reviewers
    see which category was actually present in the source data.
    """
    present = {name: 0 for name in ASSISTANCE_FACTOR_TO_INPUT}
    for factor in trial.get('mitigating_factors') or []:
        name = factor.get('factor')
        if name in present:
            present[name] = 1
            
    return {
        'assistance_limited': present['Assistance - limited'],
        'assistance_useful': present['Assistance - useful'],
        'assistance_testify': present['Assistance - testify'],
        'assistance_risk': present['Assistance - risk'],
    }

def process_trial(doc, index, trial):
    # Run the legacy model for a single trial; return an outlier row or None.
    model_input = build_model_input(trial)
    legacy_model_raw = predict_online(model_input)
    predicted_months = int(legacy_model_raw['final_sentence'])
    actual_months = get_total_months(trial.get('final_sentence'))
    difference_months = predicted_months - actual_months
    if abs(difference_months) < OUTLIER_THRESHOLD_MONTHS:
        return None
    row = {
        'neutral_citation': (doc.get('judgement') or {}).get('neutral_citation'),
        'exclude_case': bool(doc.get('exclude')),
        'trial_index': index,
        'charge_no': (trial.get('charge_type') or {}).get('charge_no'),
        'charge_name': (trial.get('charge_type') or {}).get('charge_name'),
        'drugs': format_drugs(trial),
    }
    for name, value in zip(MODEL_INPUT_COLUMNS, model_input):
        row[name] = value
        if name == 'assist_authorities':
            row.update(get_assistance_flags(trial))
    row.update({
        'legacy_model_predicted_months': predicted_months,
        'actual_months': actual_months,
        'difference_months': difference_months,
        'absolute_difference_months': abs(difference_months),
        'remarks': doc.get('remarks'),
        'legacy_model_input': model_input,
        'legacy_model_raw': legacy_model_raw,
    })
    return row

trial_tasks = [
    (doc, index, trial)
    for doc in docs
    for index, trial in enumerate(((doc.get('trials') or {}).get('trials') or []))
]

rows = []
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(process_trial, doc, index, trial) for doc, index, trial in trial_tasks]
    for future in tqdm(futures, desc='trials'):
        row = future.result()
        if row is not None:
            rows.append(row)

outlier_df = pd.DataFrame(rows).sort_values(['absolute_difference_months', 'actual_months'], ascending=[False, False]).reset_index(drop=True)
output_dir = repo_root / 'notebooks'
output_dir.mkdir(exist_ok=True)
try:
    outlier_df.to_excel(output_dir / 'model_outlier_review.xlsx', index=False)
except Exception as exc:
    print(f'Excel export skipped: {exc}')

print(f'Found {len(outlier_df)} outlier candidates using a threshold of {OUTLIER_THRESHOLD_MONTHS} months')
outlier_df.head()

judgements:   0%|          | 0/2308 [00:00<?, ?it/s]